In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path.cwd() / "datasets"


wv = {}
vector_dim = None
MAX_WORDS = 50000

# Load word vectors
with open(DATA_DIR / "dolma_300_2024_1.2M.100_combined.txt", "r", encoding="utf-8") as f:
     for i, line in enumerate(f):
        if i >= MAX_WORDS:
            break
        
        if i % 5000 == 0 and i > 0:
            print(f"已加载 {i:,} 个词...")

        parts = line.strip().split()
        if len(parts) < 2:
            continue
        
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        
        if vector_dim is None:
            vector_dim = len(vector)
            
        wv[word] = vector
print(f"\n词向量加载完成, 词汇表大小: {len(wv)}, 向量维度: {vector_dim}")

In [ ]:
def get_vec(word):
    return np.array(wv[word], dtype=float)

# Define the gender axis
gender_axis = get_vec("woman") - get_vec("man")
# Normalize the axis
gender_axis /= np.linalg.norm(gender_axis)

print("Gender axis defined and normalized.")

In [ ]:
professions = ["doctor", "nurse", "engineer", "teacher", "scientist", "secretary", "captain", "assistant"]
bias_scores = {}

for p in professions:
    if p in wv:
        v = get_vec(p)
        # Scalar projection (dot product since gender_axis is normalized)
        bias = np.dot(v, gender_axis)
        bias_scores[p] = bias
        print(f"Original bias in '{p}': {bias:.4f}")

In [ ]:
# Debiasing and plotting results
debiased_biases = []
original_biases = [bias_scores[p] for p in professions]

print("Debiasing Results:")
for p in professions:
    v = get_vec(p)
    # Remove the projection onto the gender axis
    v_debiased = v - np.dot(v, gender_axis) * gender_axis
    
    new_bias = np.dot(v_debiased, gender_axis)
    debiased_biases.append(new_bias)
    print(f"'{p}' -> New bias: {new_bias:.4e}")

# Visualization
plt.figure(figsize=(10, 5))
x = np.arange(len(professions))
plt.bar(x - 0.2, original_biases, 0.4, label='Original Bias', color='tomato')
plt.bar(x + 0.2, debiased_biases, 0.4, label='Debiased (should be ~0)', color='skyblue')

plt.xticks(x, professions)
plt.axhline(0, color='black', linewidth=0.8)
plt.title("Gender Bias in Professions: Before vs After Orthogonal Projection")
plt.ylabel("Projection onto Gender Axis")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()